<a href="https://www.kaggle.com/code/avikdas567/image2biomass-prediction?scriptVersionId=291881262" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ============================================================
# CSIRO - Image2Biomass Prediction
# ============================================================

# -----------------------------
# 1. Imports & Setup
# -----------------------------
import os
import random
import copy
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# -----------------------------
# 2. Reproducibility
# -----------------------------
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# -----------------------------
# 3. Paths
# -----------------------------
TRAIN_CSV = "/kaggle/input/csiro-biomass/train.csv"
TEST_CSV  = "/kaggle/input/csiro-biomass/test.csv"
TRAIN_DIR = "/kaggle/input/csiro-biomass/train"
TEST_DIR  = "/kaggle/input/csiro-biomass/test"

PRETRAINED_PATH = "/kaggle/input/pretrained-pytorch-models/resnet18-5c106cde.pth"

# -----------------------------
# 4. Load Data
# -----------------------------
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

train_df["image_id"] = train_df["sample_id"].str.split("__").str[0]
test_df["image_id"]  = test_df["sample_id"].str.split("__").str[0]

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

# -----------------------------
# 5. Targets
# -----------------------------
TARGETS = [
    "Dry_Green_g",
    "Dry_Dead_g",
    "Dry_Clover_g",
    "GDM_g",
    "Dry_Total_g",
]

target_to_idx = {t: i for i, t in enumerate(TARGETS)}
TARGET_WEIGHTS = torch.tensor([0.1, 0.1, 0.1, 0.2, 0.5], device=DEVICE)

# -----------------------------
# 6. Target normalization
# -----------------------------
log_targets = train_df.copy()
log_targets["log_target"] = np.log1p(log_targets["target"])

target_stats = {}
for t in TARGETS:
    vals = log_targets.loc[log_targets["target_name"] == t, "log_target"]
    target_stats[t] = (vals.mean(), vals.std() + 1e-6)

# -----------------------------
# 7. Transforms
# -----------------------------
IMG_SIZE = 224

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(0.15, 0.15, 0.15, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225],
    ),
])

test_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225],
    ),
])

# -----------------------------
# 8. Dataset
# -----------------------------
class BiomassDataset(Dataset):
    def __init__(self, df, image_dir, train=True):
        self.df = df
        self.image_dir = image_dir
        self.train = train
        self.groups = df.groupby("image_id")
        self.image_ids = list(self.groups.groups.keys())

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        rows = self.groups.get_group(image_id)

        img_path = rows.iloc[0]["image_path"]
        img = Image.open(
            os.path.join(self.image_dir, os.path.basename(img_path))
        ).convert("RGB")

        img = train_tfms(img) if self.train else test_tfms(img)

        if not self.train:
            return img, image_id

        y = torch.zeros(len(TARGETS))
        for _, r in rows.iterrows():
            mu, sd = target_stats[r["target_name"]]
            y[target_to_idx[r["target_name"]]] = (
                np.log1p(r["target"]) - mu
            ) / sd

        return img, y

# -----------------------------
# 9. DataLoaders
# -----------------------------
BATCH_SIZE = 32

train_loader = DataLoader(
    BiomassDataset(train_df, TRAIN_DIR, train=True),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    BiomassDataset(test_df, TEST_DIR, train=False),
    batch_size=BATCH_SIZE,
    shuffle=False
)

# -----------------------------
# 10. Model
# -----------------------------
class BiomassModel(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.resnet18(weights=None)

        state = torch.load(PRETRAINED_PATH, map_location="cpu", weights_only=False)
        base.load_state_dict(state)

        # Unfreeze layer2 + layer3 + layer4
        for p in base.parameters():
            p.requires_grad = False
        for layer in [base.layer2, base.layer3, base.layer4]:
            for p in layer.parameters():
                p.requires_grad = True

        self.backbone = nn.Sequential(*list(base.children())[:-1])

        self.head = nn.Sequential(
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, len(TARGETS))
        )

    def forward(self, x):
        x = self.backbone(x)
        x = x.flatten(1)
        return self.head(x)

model = BiomassModel().to(DEVICE)

# -----------------------------
# 11. EMA Helper
# -----------------------------
class EMA:
    def __init__(self, model, decay=0.995):
        self.decay = decay
        self.shadow = copy.deepcopy(model.state_dict())

    def update(self, model):
        for k, v in model.state_dict().items():
            self.shadow[k] = self.decay * self.shadow[k] + (1 - self.decay) * v

    def apply(self, model):
        model.load_state_dict(self.shadow)

ema = EMA(model)

# -----------------------------
# 12. Loss & Optimizer
# -----------------------------
criterion = nn.SmoothL1Loss(reduction="none")  # Huber loss

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=2e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=15
)

# -----------------------------
# 13. Training
# -----------------------------
EPOCHS = 12

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0

    for imgs, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        imgs = imgs.to(DEVICE)
        targets = targets.to(DEVICE)

        preds = model(imgs)
        loss = criterion(preds, targets)
        loss = (loss * TARGET_WEIGHTS).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ema.update(model)
        epoch_loss += loss.item()

    scheduler.step()
    print(f"Epoch {epoch+1} loss: {epoch_loss / len(train_loader):.4f}")

# -----------------------------
# 14. Inference (EMA weights)
# -----------------------------
ema.apply(model)
model.eval()

image_preds = {}

with torch.no_grad():
    for imgs, image_ids in tqdm(test_loader, desc="Inference"):
        imgs = imgs.to(DEVICE)
        preds = model(imgs).cpu().numpy()

        for i, img_id in enumerate(image_ids):
            out = []
            for j, t in enumerate(TARGETS):
                mu, sd = target_stats[t]
                out.append(np.expm1(preds[i, j] * sd + mu))
            image_preds[img_id] = np.array(out)

# -----------------------------
# 15. Submission
# -----------------------------
submission_targets = [
    image_preds[row["image_id"]][target_to_idx[row["target_name"]]]
    for _, row in test_df.iterrows()
]

submission = pd.DataFrame({
    "sample_id": test_df["sample_id"],
    "target": submission_targets
})

submission.to_csv("submission.csv", index=False)
print("\nSubmission saved as submission.csv")
submission.head()

Device: cuda
Train shape: (1785, 10)
Test shape : (5, 4)


Epoch 1/12: 100%|██████████| 12/12 [00:14<00:00,  1.19s/it]


Epoch 1 loss: 0.0829


Epoch 2/12: 100%|██████████| 12/12 [00:11<00:00,  1.04it/s]


Epoch 2 loss: 0.0588


...

Epoch 12/12: 100%|██████████| 12/12 [00:11<00:00,  1.04it/s]


Epoch 12 loss: 0.0346


Inference: 100%|██████████| 1/1 [00:00<00:00,  8.03it/s]


Submission saved as submission.csv


,sample_id,target
0,ID1001187975__Dry_Clover_g,2.037297
1,ID1001187975__Dry_Dead_g,7.634129
2,ID1001187975__Dry_Green_g,17.753562
3,ID1001187975__Dry_Total_g,39.765104
4,ID1001187975__GDM_g,23.846873
